# NullVector LangGraph QA Agent — PostgreSQL Backend

Build a **LangGraph ReAct agent** backed by NullVector's grounded QA in ~50 lines of code.

**Prerequisite:** Run `01_nullvector_quickstart.ipynb` first to populate PostgreSQL.

**Requirements:**
- PostgreSQL with NullVector artifacts from notebook 01
- `GROQ_API_KEYS` set in `.env` (comma-separated list for round-robin)
- Optional: `NULLVECTOR_LLM_MODEL` override (defaults to Llama 4 Scout)

See `.env.example` at the repo root for configuration.

## How it works

```
User question
    │
    ▼
┌─────────────────────────┐
│  Grounded first pass    │  NullVector QA runs BEFORE the LLM sees the question.
│  (structural retrieval  │  This gives the agent pre-verified evidence with
│   + LLM grounding)      │  source anchor provenance.
└───────────┬─────────────┘
            ▼
┌─────────────────────────┐
│  LangGraph ReAct agent  │  The LLM reviews the grounded answer and decides:
│  (LLM)                  │  - Accept it as-is, or
│  ┌────────────────────┐ │  - Call additional tools for more evidence
│  │ search_document    │ │
│  └────────────────────┘ │
└───────────┬─────────────┘
            ▼
    Final answer with citations
```

### NullVector vs naive vector RAG

| Naive vector RAG | NullVector grounded QA |
|------------------|----------------------|
| Embeds chunks → similarity search | VLM transcribes pages to Markdown → LLM synthesizes hierarchy → tree search + LLM ranking |
| No document awareness | Knows about headings, sections, page spans, and section anchors |
| Citations are approximate | Citations include exact page numbers with source anchor provenance |
| Hallucinates when evidence is thin | Explicitly says "low evidence" instead of guessing |
| Requires embedding model + vector DB | VLM + LLM, no embeddings needed |

## Setup

In [ ]:
# Uncomment to install LangGraph dependencies:
# import shutil, subprocess
# uv = shutil.which("uv")
# subprocess.run([uv, "pip", "install", "-q", "langgraph>=0.2", "langchain-openai>=0.1", "langchain-core>=0.2"], check=True)

### Imports

`NullVectorClient` handles corpus resolution and QA internally.
LangGraph/LangChain are only needed for the agent loop.

In [ ]:
from __future__ import annotations

import json
import os
import textwrap
from pathlib import Path
from typing import Annotated, TypedDict

from nullvector import NullVectorClient
from nullvector.observability import (
    DEFAULT_OBSERVABILITY_JSONL_PATH,
    configure_default_runtime_observability,
)
from nullvector.storage import PostgresStorageConfig

### Configuration & Groq Round-Robin

Same key rotation as notebook 03 — rotates across 5 Groq API keys to avoid per-key rate limits.

In [ ]:
POSTGRES_CONNINFO = os.environ.get(
    "NULLVECTOR_POSTGRES_CONNINFO",
    "postgresql://REDACTED_DB_CRED@localhost:5432/app",
)
POSTGRES_SCHEMA = os.environ.get("NULLVECTOR_POSTGRES_SCHEMA", "public")
GROQ_DEFAULT_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
AGENT_MODEL = os.environ.get("NULLVECTOR_LLM_MODEL", GROQ_DEFAULT_MODEL)
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

# Workspace — must match cookbook 01's workspace
WORKSPACE = Path.cwd() / ".artifacts" / "cookbook" / "01_workspace"


def parse_groq_api_keys(raw_keys: str) -> tuple[str, ...]:
    keys = tuple(part.strip() for part in raw_keys.split(",") if part.strip())
    if not keys:
        raise ValueError("GROQ_API_KEYS must contain at least one non-empty key.")
    return keys


def groq_key_label(slot: int, key: str) -> str:
    suffix = key[-4:] if len(key) >= 4 else key
    return f"slot-{slot:02d} (gsk_...{suffix})"


# Groq API keys for round-robin rotation.
# Set GROQ_API_KEYS as a comma-separated list in .env or shell env.
_raw_keys = os.environ.get("GROQ_API_KEYS", "")
if not _raw_keys:
    raise RuntimeError(
        "GROQ_API_KEYS env var is required. "
        "Set it in .env as a comma-separated list of Groq keys."
    )
GROQ_API_KEYS = parse_groq_api_keys(_raw_keys)
GROQ_MASKED_KEYS = tuple(
    groq_key_label(slot, key) for slot, key in enumerate(GROQ_API_KEYS, start=1)
)

groq_key_cursor = 0


def next_groq_key_selection() -> tuple[str, str]:
    global groq_key_cursor
    index = groq_key_cursor % len(GROQ_API_KEYS)
    groq_key_cursor += 1
    return GROQ_API_KEYS[index], GROQ_MASKED_KEYS[index]


### Load Corpus via NullVectorClient

The client reads its local catalog (written by notebook 03's `ingest()` + `build_description()`)
to resolve the document ID and all artifact paths. No manual Postgres queries needed.

### Pipeline Features Active in this Corpus

The corpus loaded from notebook 03 was built with these features:

- **Parallel VLM transcription** — pages transcribed in parallel windows (default: 4 concurrent)
- **Semantic anchoring** — VLM emits `SECTION_ANCHOR` markers, matched to tree nodes as `source_anchors`
- **Map-reduce hierarchy** — documents with 15+ pages use chunked parallel synthesis + merge

These features improve citation quality and hierarchy accuracy for the LangGraph agent.

In [ ]:
pg_config = PostgresStorageConfig(conninfo=POSTGRES_CONNINFO, schema=POSTGRES_SCHEMA)
OBSERVABILITY_JSONL_PATH = os.environ.get(
    "NULLVECTOR_OBSERVABILITY_JSONL_PATH",
    DEFAULT_OBSERVABILITY_JSONL_PATH,
)
runtime_logger = configure_default_runtime_observability(jsonl_path=OBSERVABILITY_JSONL_PATH)

# NullVectorClient resolves artifacts from the catalog written by notebook 01
client = NullVectorClient(WORKSPACE, storage=pg_config, logger=runtime_logger)

# Get the document ID from the client's catalog
catalog_path = WORKSPACE / ".nullvector" / "catalog.json"
if not catalog_path.exists():
    raise RuntimeError(
        f"No catalog found at {catalog_path}. Run notebook 01 first."
    )
catalog = json.loads(catalog_path.read_text())
if not catalog.get("documents"):
    raise RuntimeError("Catalog is empty. Run notebook 01 first.")

DOCUMENT_ID = next(iter(catalog["documents"]))
print(f"Document ID: {DOCUMENT_ID}")
print(f"Groq agent model: {AGENT_MODEL}")
print(f"Groq keys: {list(GROQ_MASKED_KEYS)}")

# Quick sanity check — search should return results
test_hits = client.search("introduction", document_id=DOCUMENT_ID, limit=1)
print(f"Sanity check: {len(test_hits)} hit(s) for 'introduction'")

## LangGraph Tools

Two tools wrapping `client.ask()` and `client.search()`. The key architectural choice:
**grounded first pass** runs `client.ask()` deterministically *before* the LLM sees the
question, giving the agent pre-verified evidence.

In [ ]:
from langchain_core.tools import tool


@tool
def answer_document_question(query: str) -> str:
    """Answer a question about the document using NullVector's grounded QA.

    Use this first for any content, chapter, section, page, or fact question.
    """
    response = client.ask(query, document_id=DOCUMENT_ID)
    lines = [
        f"answer_mode={response.answer_mode}",
        f"answer_strategy={response.answer_strategy}",
        f"answer={response.answer}",
    ]
    if response.citations:
        lines.append("citations:")
        for c in response.citations:
            quote = c.quote or "(no excerpt available)"
            lines.append(f"- page {c.page_label}: {quote}")
    return "\n".join(lines)


@tool
def search_document(query: str, limit: int = 5) -> str:
    """Search the document for additional evidence after the grounded QA pass."""
    hits = client.search(query, document_id=DOCUMENT_ID, limit=limit)
    if not hits:
        return "No results found for this query."
    lines: list[str] = []
    for i, hit in enumerate(hits, 1):
        unit = hit.unit
        excerpt = textwrap.shorten(unit.text or "", width=200, placeholder="...")
        lines.append(
            f"[{i}] score={hit.score:.3f} | "
            f"type={unit.unit_type.value} | "
            f"pages={unit.page_span.start_page}-{unit.page_span.end_page}\n"
            f"    {excerpt}"
        )
    return "\n\n".join(lines)


TOOLS = [answer_document_question, search_document]
TOOL_MAP = {t.name: t for t in TOOLS}
print(f"Registered tools: {list(TOOL_MAP.keys())}")

## Agent Construction

On every human turn, `call_model` runs a deterministic grounded first pass via
`client.ask()`, injects the output as a `SystemMessage`, then lets the LLM decide
whether to accept the evidence or call additional tools.

```
[human] → grounded_first_pass → [system context] → LLM → (tools?) → LLM → [answer]
```

In [ ]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages

SYSTEM_PROMPT = (
    "You are a precise document QA assistant backed by NullVector. "
    "Each user turn already includes a grounded NullVector pass before you answer. "
    "Treat that grounded context as the source of truth. "
    "If you still need more context, call search_document. "
    "If grounded tools report low evidence, say so clearly and do not invent details. "
    "Cite page numbers whenever the tool output includes them."
)


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def grounded_first_pass(query: str) -> str:
    """Run NullVector QA deterministically BEFORE the LLM sees the question."""
    return answer_document_question.invoke({"query": query})


def build_llm() -> ChatOpenAI:
    api_key, label = next_groq_key_selection()
    print(f"Groq rotation -> {label}")
    model = AGENT_MODEL.removeprefix("groq/")
    return ChatOpenAI(model=model, api_key=api_key, base_url=GROQ_BASE_URL).bind_tools(TOOLS)


def call_model(state: AgentState) -> dict[str, list[BaseMessage]]:
    messages: list[BaseMessage] = [SystemMessage(content=SYSTEM_PROMPT)]
    if state["messages"]:
        last = state["messages"][-1]
        if isinstance(last, HumanMessage):
            grounded_output = grounded_first_pass(str(last.content))
            print("Grounded first pass complete")
            messages.append(SystemMessage(content=(
                f"Grounded NullVector output:\n{grounded_output}\n"
                "Use this as the authoritative starting point for your answer."
            )))
    messages.extend(state["messages"])
    return {"messages": [build_llm().invoke(messages)]}


def execute_tools(state: AgentState) -> dict[str, list[ToolMessage]]:
    last: AIMessage = state["messages"][-1]
    results: list[ToolMessage] = []
    for tc in last.tool_calls:
        fn = TOOL_MAP.get(tc["name"])
        try:
            output = fn.invoke(tc["args"]) if fn else f"Unknown tool: {tc['name']}"
        except Exception as exc:
            output = f"Tool error: {exc}"
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"], name=tc["name"]))
    return {"messages": results}


def should_continue(state: AgentState) -> str:
    """Route to tools if the model wants more evidence, otherwise end."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", execute_tools)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

memory = MemorySaver()
app = graph.compile(checkpointer=memory)
print(f"Agent compiled with {len(GROQ_API_KEYS)} Groq keys")

In [ ]:
one_shot_counter = 0


def next_thread_id() -> str:
    global one_shot_counter
    one_shot_counter += 1
    return f"one-shot-{one_shot_counter:03d}"


def ask(question: str, *, thread_id: str | None = None, verbose: bool = False) -> str:
    """Submit a question to the agent. Omit thread_id for a fresh conversation."""
    tid = thread_id or next_thread_id()
    config = {"configurable": {"thread_id": tid}}
    result = app.invoke({"messages": [HumanMessage(content=question)]}, config)
    messages = result["messages"]
    if verbose:
        print("=" * 60)
        for msg in messages:
            role = type(msg).__name__
            content = textwrap.shorten(str(msg.content), width=120, placeholder="...")
            print(f"  [{role}] {content}")
        print("=" * 60)
    last = messages[-1]
    return last.content if isinstance(last, AIMessage) else str(last)

## Single-Turn QA

In [ ]:
answer = ask("how many pages are the preface and all that stuff is?", verbose=True)
print()
print("ANSWER:", answer)

## Multi-Turn Conversation

Q3 demonstrates **memory** — the agent uses context from Q2 to understand
"there" refers to the Sets section.

In [ ]:
THREAD = "pdf-book-demo"

print("Q1:", "What is this book about?")
a1 = ask("What is this book about?", thread_id=THREAD)
print("A1:", a1)
print()

print("Q2:", "What does the Sets section introduce?")
a2 = ask("What does the Sets section introduce?", thread_id=THREAD)
print("A2:", a2)
print()

print("Q3:", "Give one example of a set mentioned there.")
a3 = ask("Give one example of a set mentioned there.", thread_id=THREAD)
print("A3:", a3)

## Streaming Output

In [ ]:
config = {"configurable": {"thread_id": "streaming-demo"}}
payload = {"messages": [HumanMessage(content="What does the Sets section introduce?")]}

for event in app.stream(payload, config, stream_mode="values"):
    last = event["messages"][-1]
    role = type(last).__name__
    if isinstance(last, AIMessage) and last.tool_calls:
        for tc in last.tool_calls:
            print(f"  [{role}] calling {tc['name']}({tc['args']})")
    elif isinstance(last, ToolMessage):
        excerpt = textwrap.shorten(last.content, width=100, placeholder="...")
        print(f"  [{role}] {last.name} -> {excerpt}")
    elif isinstance(last, AIMessage):
        print(f"\n  [{role}] FINAL: {last.content}")

## Notes

- This notebook requires live Groq credentials — there is no noop fallback because the LangGraph agent needs a live LLM
- The corpus is loaded via `NullVectorClient`'s catalog, written by notebook 03's `ingest()` + `build_description()`
- Both notebooks share the same workspace path and `NULLVECTOR_NOTEBOOK_RUN_SUFFIX` (default `v5`)
- `client.ask()` handles all three QA modes (document_summary, focused_lookup, low_evidence) automatically
- Per-call rotation only prints masked key labels — notebook traces never echo full Groq secrets
- The corpus benefits from parallel VLM transcription, semantic anchoring (section anchors matched to tree nodes), and map-reduce hierarchy synthesis for large documents
- Source anchors on tree nodes provide spatial provenance — the agent's citations trace back to exact positions in the VLM-transcribed Markdown